# Data Preprocessing

This notebook handles data cleaning, feature engineering, and preprocessing for model training.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
import pickle

In [2]:
# Load the dataset from Excel file
df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx')

# Standardize column names
df = df.rename(columns={
    'Gender': 'gender',
    'Senior Citizen': 'SeniorCitizen',
    'Tenure Months': 'tenure',
    'Phone Service': 'PhoneService',
    'Multiple Lines': 'MultipleLines',
    'Internet Service': 'InternetService',
    'Online Security': 'OnlineSecurity',
    'Online Backup': 'OnlineBackup',
    'Device Protection': 'DeviceProtection',
    'Tech Support': 'TechSupport',
    'Streaming TV': 'StreamingTV',
    'Streaming Movies': 'StreamingMovies',
    'Paperless Billing': 'PaperlessBilling',
    'Payment Method': 'PaymentMethod',
    'Monthly Charges': 'MonthlyCharges',
    'Total Charges': 'TotalCharges',
    'Churn Label': 'Churn'
})

# Drop unnecessary columns
columns_to_drop = ['Count', 'Country', 'State', 'City', 'Zip Code', 
                   'Lat Long', 'Latitude', 'Longitude', 'CustomerID',
                   'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')

print(f"Original dataset shape: {df.shape}")
df.head()

Original dataset shape: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes


## Data Cleaning

In [3]:
# Handle missing values
print("Missing values before cleaning:")
print(df.isnull().sum())

# Example: Fill numerical missing values with median
numerical_cols = df.select_dtypes(include=[np.number]).columns
imputer_num = SimpleImputer(strategy='median')
df[numerical_cols] = imputer_num.fit_transform(df[numerical_cols])

# Example: Fill categorical missing values with mode
categorical_cols = df.select_dtypes(include=['object']).columns
imputer_cat = SimpleImputer(strategy='most_frequent')
df[categorical_cols] = imputer_cat.fit_transform(df[categorical_cols])

print("\nMissing values after cleaning:")
print(df.isnull().sum())

Missing values before cleaning:
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Missing values after cleaning:
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [4]:
# Remove duplicates
print(f"Duplicates: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Shape after removing duplicates: {df.shape}")

Duplicates: 22
Shape after removing duplicates: (7021, 20)


## Feature Engineering

In [5]:
# Create new features (example)
# df['feature_ratio'] = df['feature1'] / (df['feature2'] + 1)
# df['is_high_value'] = (df['TotalCharges'] > df['TotalCharges'].median()).astype(int)

print("Feature engineering completed")

Feature engineering completed


## Encoding Categorical Variables

In [6]:
# Separate features and target
if 'Churn' in df.columns:
    X = df.drop('Churn', axis=1)
    y = df['Churn']
    
    # Encode target variable if it's categorical
    if y.dtype == 'object':
        le = LabelEncoder()
        y = le.fit_transform(y)
        print(f"Target classes: {le.classes_}")
else:
    X = df.copy()
    y = None

Target classes: ['No' 'Yes']


In [7]:
# Encode categorical features
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical features: {categorical_features}")
print(f"Numerical features: {numerical_features}")

# One-hot encoding for categorical variables
X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=True)
print(f"\nShape after encoding: {X_encoded.shape}")

Categorical features: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges']
Numerical features: ['tenure', 'MonthlyCharges']

Shape after encoding: (7021, 6559)


## Train-Test Split

In [8]:
# Split data into training and testing sets
if y is not None:
    X_train, X_test, y_train, y_test = train_test_split(
        X_encoded, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"Training set size: {X_train.shape}")
    print(f"Testing set size: {X_test.shape}")
    print(f"\nTarget distribution in training set:")
    print(pd.Series(y_train).value_counts(normalize=True))

Training set size: (5616, 6559)
Testing set size: (1405, 6559)

Target distribution in training set:
0    0.735577
1    0.264423
Name: proportion, dtype: float64


## Feature Scaling

In [9]:
# Standardize numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("Feature scaling completed")

Feature scaling completed


## Save Processed Data

In [10]:
# Save processed data
X_train_scaled.to_csv('../data/processed/X_train.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test.csv', index=False)
pd.DataFrame(y_train, columns=['Churn']).to_csv('../data/processed/y_train.csv', index=False)
pd.DataFrame(y_test, columns=['Churn']).to_csv('../data/processed/y_test.csv', index=False)

# Save scaler for future use
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Processed data saved successfully!")

Processed data saved successfully!
